In this project, I focused on applying advanced computer vision to environmental monitoring. Specifically, I fine-tuned the Swin V2 Transformer for fallen tree segmentation in aerial imagery. This task is crucial for disaster recovery and forest management, yet challenging due to irregular tree shapes and cluttered backgrounds. By leveraging Swin V2’s hierarchical attention and integrating it into a Mask R-CNN framework, I built a pipeline that achieves strong segmentation accuracy and produces visual overlays for validation. My work shows how transformer-based models can support ecological monitoring and real-world forestry applications. If anyone interested to replicate and improve the model further, and need the fallen tree segmentation dataset, please contact me at: *sakhawat3003@gmail.com*

## *Import Necessary Libraries*

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os, cv2, json, random, csv
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork, MultiScaleRoIAlign, box_iou
from collections import OrderedDict
from tqdm import tqdm
import timm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path="/content/drive/MyDrive/Colab Notebooks/swinv2_fallen_tree_segmentation/tree_data_for_instance_segmentation"

In [ ]:
os.listdir(data_path)

['McCormick_25Apr01_0_0.json',
 'McCormick_25Apr01_0_2.json',
 'McCormick_25Apr01_0_1.tif',
 'McCormick_25Apr01_0_1.json',
 'McCormick_25Apr01_0_0.tif',
 'McCormick_25Apr01_0_2.tif',
 'McCormick_25Apr01_0_3.json',
 'McCormick_25Apr01_0_3.tif',
 'McCormick_25Apr01_1_0.json',
 'McCormick_25Apr01_0_4.tif',
 'McCormick_25Apr01_0_4.json',
 'McCormick_25Apr01_1_0.tif',
 'McCormick_25Apr01_1_1.json',
 'McCormick_25Apr01_1_2.json',
 'McCormick_25Apr01_1_1.tif',
 'McCormick_25Apr01_1_3.tif',
 'McCormick_25Apr01_1_3.json',
 'McCormick_25Apr01_1_2.tif',
 'McCormick_25Apr01_1_4.tif',
 'McCormick_25Apr01_1_4.json',
 'McCormick_25Apr01_2_0.json',
 'McCormick_25Apr01_2_1.tif',
 'McCormick_25Apr01_2_1.json',
 'McCormick_25Apr01_2_2.json',
 'McCormick_25Apr01_2_0.tif',
 'McCormick_25Apr01_2_3.json',
 'McCormick_25Apr01_2_2.tif',
 'McCormick_25Apr01_2_3.tif',
 'McCormick_25Apr01_3_1.json',
 'McCormick_25Apr01_3_3.tif',
 'McCormick_25Apr01_3_1.tif',
 'McCormick_25Apr01_3_3.json',
 'McCormick_25Apr01_4_2.

## *Explore Aerial Image with Masked Overlay*

In [ ]:
img_path = os.path.join(data_path, "McCormick_25Apr01_0_1.tif")   # image file
json_path = os.path.join(data_path, "McCormick_25Apr01_0_1.json") # annotation file
# Load image
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Load annotation
with open(json_path) as f:
    ann = json.load(f)

# Create blank mask
h, w = ann["imageHeight"], ann["imageWidth"]
mask = np.zeros((h, w), dtype=np.int32)

# Fill polygons with unique IDs
for idx, shape in enumerate(ann["shapes"], start=1):
    pts = np.array(shape["points"], dtype=np.int32)
    cv2.fillPoly(mask, [pts], idx)

# Overlay visualization
overlay = img.copy()
for uid in np.unique(mask):
    if uid == 0:  # skip background
        continue
    color = np.random.randint(0, 255, size=3)
    overlay[mask == uid] = color

# Show results
plt.figure(figsize=(12,6))
plt.subplot(1,2,1); plt.imshow(img); plt.title("Original Image")
plt.subplot(1,2,2); plt.imshow(overlay); plt.title("Mask Overlay");

## *Data Preprocessing*

In [ ]:
# Path to the dataset
data_path = "/content/drive/MyDrive/Colab Notebooks/tree_data_for_instance_segmentation"

# Create a folder to save masks
masks_dir = os.path.join(data_path, "masks")
os.makedirs(masks_dir, exist_ok=True)

# Normalize label variations
label_map = {
    "fallen_tree": "fallen_tree",
    "fallen_trees": "fallen_tree",
    "Fallen_Tree": "fallen_tree",
    "FALLEN_TREE": "fallen_tree"
}

def json_to_mask(json_path, save_path=None, max_size=2048):
    with open(json_path) as f:
        ann = json.load(f)

    h, w = ann["imageHeight"], ann["imageWidth"]
    mask = np.zeros((h, w), dtype=np.int32)

    for idx, shape in enumerate(ann["shapes"], start=1):
        # normalize label
        raw_label = shape["label"]
        label = label_map.get(raw_label, raw_label).lower()
        if label != "fallen_tree":
            continue
        pts = np.array(shape["points"], dtype=np.int32)
        cv2.fillPoly(mask, [pts], idx)

    # --- Resize mask to max_size and enforce multiples of 32 ---
    scale = max_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    new_w = max(32, int(np.ceil(new_w / 32)) * 32)
    new_h = max(32, int(np.ceil(new_h / 32)) * 32)
    mask_resized = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    if save_path:
        np.save(save_path, mask_resized)
    return mask_resized

# Collect all JSON files
json_files = [f for f in os.listdir(data_path) if f.endswith(".json")]

# Process all JSON files
for jf in tqdm(json_files):
    json_path = os.path.join(data_path, jf)
    mask_path = os.path.join(masks_dir, jf.replace(".json", ".npy"))
    json_to_mask(json_path, save_path=mask_path)

print(f"Processed {len(json_files)} masks.")


100%|██████████| 18/18 [00:50<00:00,  2.83s/it]

Processed 18 masks.


## *Train Set and Validation Set*

In [ ]:
# Create train/val split
train_files, val_files = train_test_split(json_files, test_size=0.2, random_state=42)

# Write train.txt
with open(os.path.join(data_path, "train.txt"), "w") as f:
    for fn in train_files:
        f.write(fn.replace(".json", "") + "\n")

# Write val.txt
with open(os.path.join(data_path, "val.txt"), "w") as f:
    for fn in val_files:
        f.write(fn.replace(".json", "") + "\n")

print(f"Train: {len(train_files)}, Val: {len(val_files)}")

Train: 16, Val: 2


## *Check Train Files and Validation Files*

In [ ]:
with open(os.path.join(data_path, "train.txt")) as f:
    train_files = [line.strip() for line in f.readlines()]
print(train_files)

['McCormick_25Apr01_0_3', 'McCormick_25Apr01_2_3', 'McCormick_25Apr01_4_2', 'McCormick_25Apr01_3_3', 'McCormick_25Apr01_2_1', 'McCormick_25Apr01_0_2', 'McCormick_25Apr01_1_4', 'McCormick_25Apr01_4_3', 'McCormick_25Apr01_0_4', 'McCormick_25Apr01_2_2', 'McCormick_25Apr01_1_2', 'McCormick_25Apr01_2_0', 'McCormick_25Apr01_3_1', 'McCormick_25Apr01_1_1']


In [ ]:
with open(os.path.join(data_path, "val.txt")) as f:
    val_files = [line.strip() for line in f.readlines()]
print(val_files)

['McCormick_25Apr01_0_0', 'McCormick_25Apr01_0_1', 'McCormick_25Apr01_1_3', 'McCormick_25Apr01_1_0']


## GPU or CPU

In [ ]:
# Automatically select GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

Using device: cuda
GPU name: Tesla T4


## Swin V2 Transformer Model

### *Create Dataset and Define the Model*

In [ ]:
# --- Resize Utility (must be defined before the dataset class) ---
import numpy as np

def resize_keep_ratio(img, mask, max_size=2048):
    h, w = img.shape[:2]
    scale = max_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)

    # --- enforce multiples of 32 (round UP) ---
    new_w = max(32, int(np.ceil(new_w / 32)) * 32)
    new_h = max(32, int(np.ceil(new_h / 32)) * 32)

    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    mask_resized = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    return img_resized, mask_resized

# --- Dataset Class ---
class TreeSegmentationDataset(Dataset):
    def __init__(self, data_path, file_list, max_size=2048, img_ext=".tif"):
        """
        Args:
            data_path (str): Root folder containing images and 'masks' subfolder.
            file_list (list[str]): List of base filenames (without extension).
            max_size (int): Max dimension for resizing while keeping aspect ratio.
            img_ext (str): Image file extension (default '.tif').
        """
        self.data_path = data_path
        self.file_list = file_list
        self.max_size = max_size
        self.img_ext = img_ext

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        base_name = self.file_list[idx]

        # --- Safe image loading ---
        img_path = os.path.join(self.data_path, base_name + self.img_ext)
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Failed to read image (bad path or extension?): {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # --- Safe mask loading ---
        mask_path = os.path.join(self.data_path, "masks", base_name + ".npy")
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"Mask not found: {mask_path}")
        mask = np.load(mask_path)

        # --- Resize to multiple of 32 for Swin ---
        img, mask = resize_keep_ratio(img, mask, max_size=self.max_size)

        # --- Extract object masks and bounding boxes ---
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[obj_ids != 0]  # exclude background
        masks = mask == obj_ids[:, None, None]

        boxes, valid_masks = [], []
        for i in range(len(obj_ids)):
            pos = np.where(masks[i])
            if pos[0].size == 0 or pos[1].size == 0:
                continue
            xmin, xmax = np.min(pos[1]), np.max(pos[1])
            ymin, ymax = np.min(pos[0]), np.max(pos[0])
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
                valid_masks.append(masks[i])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            masks = torch.zeros((0, mask.shape[0], mask.shape[1]), dtype=torch.uint8)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            masks = torch.as_tensor(np.stack(valid_masks, axis=0), dtype=torch.bool)
            labels = torch.ones((len(boxes),), dtype=torch.int64)  # all foreground class

        # --- Convert image to tensor ---
        img_tensor = torch.as_tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor(idx)
        }
        return img_tensor, target

# ---------------- SwinV2 + FPN Backbone ----------------
class SwinV2FPNBackbone(nn.Module):
    def __init__(self, model_name='swinv2_tiny_window8_256', pretrained=True, fpn_out_channels=256):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=(0,1,2,3),
            img_size=None,
            dynamic_img_size=True,
            strict_img_size=False
        )

        # Get actual channel sizes from feature_info, not from a dummy input's shape[1]
        self.in_channels_list = self.backbone.feature_info.channels()
        print("Actual feature channels from feature_info:", self.in_channels_list)

        # Projection layers to unify channel depth
        self.input_proj = nn.ModuleList([
            nn.Conv2d(c, fpn_out_channels, kernel_size=1) for c in self.in_channels_list
        ])

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[fpn_out_channels]*len(self.in_channels_list),
            out_channels=fpn_out_channels
        )
        self.out_channels = fpn_out_channels

    def forward(self, x):
        feats = self.backbone(x)
        # Permute features from (N, H, W, C) to (N, C, H, W) for Conv2d layers
        feats_permuted = [f.permute(0, 3, 1, 2) for f in feats]
        feats_proj = [proj(f_permuted) for proj, f_permuted in zip(self.input_proj, feats_permuted)]
        feats_dict = OrderedDict({str(i): f for i, f in enumerate(feats_proj)})
        return self.fpn(feats_dict)

# ---------------- Evaluation Utilities ----------------
def evaluate_boxes(model, data_loader, device, threshold=0.5):
    model.eval(); iou_scores = []
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            for output, target in zip(outputs, targets):
                keep = output["scores"].cpu() > threshold
                pred_boxes = output["boxes"].cpu()[keep]
                gt_boxes = target["boxes"].cpu()
                if len(gt_boxes) == 0 or len(pred_boxes) == 0: continue
                ious = box_iou(pred_boxes, gt_boxes)
                max_ious, _ = ious.max(dim=1)
                iou_scores.extend(max_ious.tolist())
    return np.mean(iou_scores) if iou_scores else 0.0

def evaluate_masks(model, data_loader, device, threshold=0.5):
    model.eval(); iou_scores = []
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            for output, target in zip(outputs, targets):
                keep = output["scores"].cpu() > threshold
                pred_masks = (output["masks"].cpu()[keep] > 0.5).squeeze(1)
                gt_masks = target["masks"].cpu()
                if len(gt_masks) == 0 or len(pred_masks) == 0: continue
                for pm in pred_masks:
                    ious = []
                    for gm in gt_masks:
                        intersection = (pm & gm).sum().item()
                        union = (pm | gm).sum().item()
                        if union > 0: ious.append(intersection / union)
                    if ious: iou_scores.append(max(ious))
    return np.mean(iou_scores) if iou_scores else 0.0

def plot_training_curves(train_losses, box_ious, mask_ious, save_path="training_summary.png"):
    epochs = range(1, len(train_losses)+1)
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.plot(epochs, train_losses); plt.title("Train Loss")
    plt.subplot(1,3,2); plt.plot(epochs, box_ious); plt.title("Box IoU")
    plt.subplot(1,3,3); plt.plot(epochs, mask_ious); plt.title("Mask IoU")
    plt.tight_layout(); plt.savefig(save_path); plt.close()

# ---------------- Training Loop ----------------
import torch
from tqdm import tqdm

def train_model(model, train_loader, val_loader, optimizer, scheduler, device,
                num_epochs=50, grad_clip=5.0, save_best=True, save_last=True,
                early_stop_patience=None, grad_accum_steps=1):
    train_losses, val_losses, box_ious, mask_ious, lrs = [], [], [], [], []
    best_box_iou = 0.0   # <-- track best Box IoU
    patience_counter = 0

    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        print(f"DEBUG: Model training status at start of epoch {epoch+1}: {model.training}")
        epoch_loss = 0.0
        optimizer.zero_grad()

        for step, (images, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

            loss_output = model(images, targets)
            if isinstance(loss_output, dict):
                loss_dict = loss_output
            elif isinstance(loss_output, list):
                raise RuntimeError("Unexpected model output type in training mode.")
            else:
                raise TypeError(f"Unexpected return type from model: {type(loss_output)}")

            losses = sum(loss for loss in loss_dict.values())
            losses.backward()

            # Gradient accumulation
            if (step + 1) % grad_accum_steps == 0:
                if grad_clip is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                optimizer.zero_grad()

            epoch_loss += losses.item()

        avg_train_loss = epoch_loss / max(1, len(train_loader))

        # --- Validation ---
        model.eval()
        val_loss, val_batches = 0.0, 0
        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k,v in t.items()} for t in targets]
                _ = model(images, targets)
                val_batches += 1

        avg_val_loss = 0.0  # placeholder

        mean_box_iou = evaluate_boxes(model, val_loader, device)
        mean_mask_iou = evaluate_masks(model, val_loader, device)

        # --- Logging ---
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, "
              f"Box IoU={mean_box_iou:.4f}, Mask IoU={mean_mask_iou:.4f}, "
              f"LR={current_lr:.6f}")

        # --- Save best model (based on Box IoU) ---
        if save_best and mean_box_iou > best_box_iou:
            best_box_iou = mean_box_iou
            torch.save(model.state_dict(), "best_model.pth")
            print(f"Saved new best model (Box IoU={best_box_iou:.4f})")

        # --- Save last model ---
        if save_last:
            torch.save(model.state_dict(), "last_model.pth")

        # --- Early stopping (based on Box IoU) ---
        if early_stop_patience is not None:
            if mean_box_iou <= best_box_iou:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    print("Early stopping triggered.")
                    break
            else:
                patience_counter = 0

        # --- Update scheduler ---
        scheduler.step()

        # --- Store history ---
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        box_ious.append(mean_box_iou)
        mask_ious.append(mean_mask_iou)
        lrs.append(current_lr)

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "box_ious": box_ious,
        "mask_ious": mask_ious,
        "lrs": lrs
    }


# Data Loading
data_path = "/content/drive/MyDrive/Colab Notebooks/swinv2_fallen_tree_segmentation/tree_data_for_instance_segmentation"
with open(os.path.join(data_path, "train.txt")) as f: train_files = [line.strip() for line in f.readlines()]
with open(os.path.join(data_path, "val.txt")) as f: val_files = [line.strip() for line in f.readlines()]
train_dataset = TreeSegmentationDataset(data_path, train_files)
val_dataset   = TreeSegmentationDataset(data_path, val_files)
def collate_fn(batch): return tuple(zip(*batch))
# ---------------- DataLoaders ----------------
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# ---------------- Build Model ----------------
import torchvision
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.rpn import AnchorGenerator

# Instantiate your custom backbone
backbone = SwinV2FPNBackbone(
    model_name='swinv2_tiny_window8_256',
    pretrained=True,
    fpn_out_channels=256
)

# Anchor generator for FPN levels [0,1,2,3]
anchor_generator = AnchorGenerator(
    sizes=((32,), (64,), (128,), (256,)),
    aspect_ratios=((0.5, 1.0, 2.0),) * 4
)

# ROI pooler for FPN
roi_pooler = torchvision.ops.MultiScaleRoIAlign(
    featmap_names=[str(i) for i in range(len(backbone.in_channels_list))],
    output_size=7,
    sampling_ratio=2
)

# Mask pooler for FPN
mask_roi_pooler = torchvision.ops.MultiScaleRoIAlign(
    featmap_names=[str(i) for i in range(len(backbone.in_channels_list))],
    output_size=14,
    sampling_ratio=2
)

# Build Mask R-CNN
model = MaskRCNN(
    backbone,
    num_classes=2,   # <-- set this to your dataset’s number of classes
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler,
    mask_roi_pool=mask_roi_pooler
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Actual feature channels from feature_info: [96, 192, 384, 768]


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): SwinV2FPNBackbone(
    (backbone): FeatureListNet(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      )
      (layers_0): SwinTransformerV2Stage(
        (downsample): Identity()
        (blocks): ModuleList(
          (0): SwinTransformerV2Block(
            (attn): WindowAttention(
              (cpb_mlp): Sequential(
                (0): Linear(in_features=2, out_features=512, bias=True)
                (1): ReLU(inplace=True)
                (2): Linear(in_features=512, out_features=3, bias=False)
              )
              (qkv): Linear(in_features=96, out_features=288, bias=False)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): 

### *Optimizer*

In [ ]:
# Define how many epochs you want to train
num_epochs = 20   # or any integer you prefer

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=5e-5, weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs, eta_min=1e-6
)

### *Train Model*

In [ ]:
# ---------------- Run Training ----------------
train_model(model, train_loader, val_loader, optimizer, scheduler, device, num_epochs=20)

DEBUG: Model training status at start of epoch 1: True


Epoch 1/20 [Val]: 100%|██████████| 4/4 [00:07<00:00,  1.79s/it]


Epoch 1/20 | Train Loss=3.7596, Val Loss=0.0000, Box IoU=0.0000, Mask IoU=0.0000, LR=0.000050
DEBUG: Model training status at start of epoch 2: True


Epoch 2/20 [Val]: 100%|██████████| 4/4 [00:07<00:00,  1.89s/it]


Epoch 2/20 | Train Loss=2.5412, Val Loss=0.0000, Box IoU=0.0000, Mask IoU=0.0000, LR=0.000050
DEBUG: Model training status at start of epoch 3: True


...

Epoch 19/20 [Val]: 100%|██████████| 4/4 [00:06<00:00,  1.73s/it]


Epoch 19/20 | Train Loss=1.7279, Val Loss=0.0000, Box IoU=0.4247, Mask IoU=0.1521, LR=0.000002
DEBUG: Model training status at start of epoch 20: True


Epoch 20/20 [Val]: 100%|██████████| 4/4 [00:07<00:00,  1.89s/it]


Epoch 20/20 | Train Loss=1.6946, Val Loss=0.0000, Box IoU=0.4248, Mask IoU=0.1423, LR=0.000001


{'train_losses': [3.759575997080122,
  2.541160592011043,
  2.305662623473576,
  2.3329038875443593,
  2.1936477763312205,
  2.1716705432959964,
  2.077839140381132,
  1.9870013850075858,
  2.004072598048619,
  1.9602545925549097,
  1.9098807041134154,
  1.8553469010761805,
  1.8565697254879134,
  1.7819829668317522,
  1.7480663806200027,
  1.7154250336544854,
  1.6904593610337801,
  1.6941055485180445,
  1.7279131361948592,
  1.6945854457361358],
 'val_losses': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'box_ious': [0.0,
  0.0,
  0.0,
  0.0,
  np.float64(0.11449963517225115),
  np.float64(0.41521349581687345),
  np.float64(0.5662291049957275),
  np.float64(0.3707486456976487),
  np.float64(0.3352312174889759),
  np.float64(0.37440662999543245),
  np.float64(0.36434726354244323),
  np.float64(0.39094708622136015),
  np.float64(0.40753882008676345),
  np.float64(0.4097652580150787),
  np.fl

### *Load the best model without saving to drive*

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): SwinV2FPNBackbone(
    (backbone): FeatureListNet(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      )
      (layers_0): SwinTransformerV2Stage(
        (downsample): Identity()
        (blocks): ModuleList(
          (0): SwinTransformerV2Block(
            (attn): WindowAttention(
              (cpb_mlp): Sequential(
                (0): Linear(in_features=2, out_features=512, bias=True)
                (1): ReLU(inplace=True)
                (2): Linear(in_features=512, out_features=3, bias=False)
              )
              (qkv): Linear(in_features=96, out_features=288, bias=False)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): 

### *Prepare Validation set for evaluation*

In [ ]:
val_dataset = TreeSegmentationDataset(data_path, val_files)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

### *Evaluation on the Validation set*

In [ ]:
# Box IoU evaluation
mean_box_iou = evaluate_boxes(model, val_loader, device)

# Mask IoU evaluation
mean_mask_iou = evaluate_masks(model, val_loader, device)

print(f"Validation Results -> Box IoU: {mean_box_iou:.4f}, Mask IoU: {mean_mask_iou:.4f}")

Validation Results -> Box IoU: 0.5652, Mask IoU: 0.0020


### *Load the Best saved Model and Evaluate*

In [ ]:
# Path to the saved model
model_path = "/content/drive/MyDrive/Colab Notebooks/swinv2_fallen_tree_segmentation/swinv2_best_student.pth"

# Load checkpoint
checkpoint = torch.load(model_path, map_location=device)  # device = 'cuda' or 'cpu'

# Load into the model
model.load_state_dict(checkpoint)
model.to(device)
model.eval()

MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): SwinV2FPNBackbone(
    (backbone): FeatureListNet(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      )
      (layers_0): SwinTransformerV2Stage(
        (downsample): Identity()
        (blocks): ModuleList(
          (0): SwinTransformerV2Block(
            (attn): WindowAttention(
              (cpb_mlp): Sequential(
                (0): Linear(in_features=2, out_features=512, bias=True)
                (1): ReLU(inplace=True)
                (2): Linear(in_features=512, out_features=3, bias=False)
              )
              (qkv): Linear(in_features=96, out_features=288, bias=False)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): 

### *Prepare the Validation Set*

In [ ]:
val_dataset = TreeSegmentationDataset(data_path, val_files)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

### *Run Evaluation*

In [ ]:
# Box IoU evaluation
mean_box_iou = evaluate_boxes(model, val_loader, device)

# Mask IoU evaluation
mean_mask_iou = evaluate_masks(model, val_loader, device)

print(f"Validation Results -> Box IoU: {mean_box_iou:.4f}, Mask IoU: {mean_mask_iou:.4f}")

Validation Results -> Box IoU: 0.7389, Mask IoU: 0.4711


### Generate overlays (original + GT + prediction)

In [ ]:
def visualize_gt_and_pred(image, gt_masks, pred_masks, save_path):
    plt.figure(figsize=(18,6))

    # Original
    plt.subplot(1,3,1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis("off")

    # Ground truth overlay
    overlay_gt = image.copy()
    for mask in gt_masks:
        color = [random.randint(0,255) for _ in range(3)]
        mask_np = mask.cpu().numpy().astype(bool)
        overlay_gt[mask_np] = (0.5*overlay_gt[mask_np] + 0.5*np.array(color)).astype(np.uint8)
    plt.subplot(1,3,2)
    plt.imshow(overlay_gt)
    plt.title("Ground Truth Masks")
    plt.axis("off")

    # Prediction overlay
    overlay_pred = image.copy()
    # 🔑 Iterate over all predicted masks
    for mask in pred_masks:
        color = [random.randint(0,255) for _ in range(3)]
        mask_np = mask.cpu().numpy().astype(bool)
        overlay_pred[mask_np] = (0.5*overlay_pred[mask_np] + 0.5*np.array(color)).astype(np.uint8)
    plt.subplot(1,3,3)
    plt.imshow(overlay_pred)
    plt.title("Predicted Masks")
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

### Run evaluation and save visual maps

In [ ]:
os.makedirs("visual_maps", exist_ok=True)

with torch.no_grad():
    for idx, (images, targets) in enumerate(val_loader):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for img_tensor, target, output in zip(images, targets, outputs):
            image_id = target["image_id"].item()
            img_np = (img_tensor.cpu().permute(1,2,0).numpy()*255).astype(np.uint8)

            gt_masks = target["masks"]
            keep = output["scores"].cpu() > 0.2
            pred_masks = (output["masks"].cpu()[keep] > 0.5).squeeze(1) if keep.sum() > 0 else []

            save_path = f"visual_maps/{image_id}_comparison.png"
            visualize_gt_and_pred(img_np, gt_masks, pred_masks, save_path)

### Save the *visual_maps* folder inside *swinv2_fallen_tree_segmentation*

In [ ]:
!cp -r /content/visual_maps "/content/drive/MyDrive/Colab Notebooks/swinv2_fallen_tree_segmentation/"